# 01.03b 線形単回帰の実データでの評価 (NHANESデータセット)

このノートブックでは、米国の国民健康栄養調査（NHANES）のデータを使用し、大腿骨長から身長を予測する単回帰モデルを構築します。
ここでは特に、**実データにおける「評価の作法」**（訓練・テスト分割、RMSE、ベースライン比較）に焦点を当てます。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from IPython.display import display, Latex

# グラフのスタイル設定
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["font.family"] = "sans-serif"

## 1. データの準備
大腿骨長（UpperLegLength）を特徴量 $x$、身長（StandingHeight）をターゲット $y$ として抽出します。

In [ ]:
# NHANESデータの読み込み（オンラインより）
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.xpt"
df_raw = pd.read_sas(url)

# 欠損値の除去と必要な列の抽出
df = df_raw.query("BMXHT.notnull() and BMXLEG.notnull()")[["BMXHT", "BMXLEG"]].copy()
df.rename(columns={"BMXHT": "StandingHeight", "BMXLEG": "UpperLegLength"}, inplace=True)

print(f"総サンプル数: {len(df)}")
df.head()

## 2. オフライン評価の開始：データを分割する
「未来のデータ」のふりをするために、手元のデータを分割（Splitting）します。

In [ ]:
X = df[["UpperLegLength"]]
y = df["StandingHeight"]

# 訓練データ 80%、テストデータ 20% に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"訓練データ数 (Train): {len(X_train)}")
print(f"テストデータ数 (Test) : {len(X_test)}")

## 3. モデルの学習
訓練データのみを使って、最小二乗法でパラメーターを推定します。

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

w = model.coef_[0]
b = model.intercept_

display(Latex(f"学習されたモデル: $\\hat{{y}} = {b:.2f} + {w:.2f} \\cdot x$"))

## 4. 性能評価（RMSEと相対性能）

テストデータに対する予測誤差を計算します。また、比較のために「常に平均値を答えるベースライン」の誤差も計算し、モデルがどれくらい価値があるか（相対性能）を確認します。

In [ ]:
# モデルによる予測
y_pred = model.predict(X_test)
rmse_model = np.sqrt(mean_squared_error(y_test, y_pred))

# ベースライン（平均値予測）による予測
y_mean = np.full_like(y_test, y_train.mean()) # 訓練データの平均を予測として使う
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_mean))

print(f"ベースラインの誤差 (平均予測) RMSE: {rmse_baseline:.2f} cm")
print(f"構築したモデルの誤差           RMSE: {rmse_model:.2f} cm")
print(f"相対的な改善率: {(1 - rmse_model/rmse_baseline):.1%}")

## 5. 結果の可視化

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.3, label="テストデータ (実地データ)", color="gray")
plt.plot(X_test, y_pred, color="red", linewidth=3, label="モデルの予測線 (学習済み)")
plt.xlabel("大腿骨長 (Upper Leg Length) [cm]")
plt.ylabel("身長 (Standing Height) [cm]")
plt.title(f"NHANES 身長予測結果 (RMSE: {rmse_model:.2f} cm)")
plt.legend()
plt.show()